# 🚀 Huấn luyện YOLOv8 SMALL — Nhận diện Xe cộ
- Dataset: **VEHICLE_DATASET_YOLO** (đã làm sạch, 0 leakage, đã augment sẵn ở bước chuẩn bị).
- **KHÔNG augment thêm** ở đây — chỉ train với bộ thông số chuẩn.
- Model: `yolov8s.pt` | Batch gợi ý: `16`

**Hướng dẫn:** Bật GPU `T4 x2` → Add Data folder chứa `VEHICLE_DATASET_YOLO` → Run All.

## 1. Cài đặt thư viện & Kiểm tra GPU

In [1]:
!pip install ultralytics -q
import torch
from ultralytics import YOLO
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('So GPU:', torch.cuda.device_count(), '|', torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 754.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.

## 2. Tự dò dataset & tạo data.yaml
Tự tìm folder `VEHICLE_DATASET_YOLO` trong `/kaggle/input` — không cần sửa đường dẫn tay.

In [2]:
import glob, yaml, os

BATCH = 8              # chinh nho lai neu GPU yeu (OOM)

hits = glob.glob('/kaggle/input/**/VEHICLE_DATASET_YOLO', recursive=True)
assert hits, 'Chua Add Data: khong thay VEHICLE_DATASET_YOLO trong /kaggle/input'
DATASET_PATH = hits[0]

cfg = {'path': DATASET_PATH, 'train': 'train/images', 'val': 'val/images',
       'test': 'test/images', 'nc': 4,
       'names': {0: 'Bus', 1: 'Motorcycle', 2: 'Car', 3: 'Truck'}}
YAML_PATH = '/kaggle/working/data.yaml'
yaml.dump(cfg, open(YAML_PATH, 'w'), sort_keys=False)

print('Dataset:', DATASET_PATH)
for s in ['train', 'val', 'test']:
    print(' ', s, len(glob.glob(os.path.join(DATASET_PATH, s, 'images', '*'))), 'anh')

Dataset: /kaggle/input/<dataset>/VEHICLE_DATASET_YOLO
  train 3357 anh
  val 324 anh
  test 167 anh


## 3. Tải mô hình YOLOv8 SMALL

In [3]:
model = YOLO('yolov8s.pt')   # pretrained
print('[OK] Loaded', model.model_name if hasattr(model,'model_name') else '')

[OK] Loaded yolov8s.pt


## 4. Huấn luyện
Bộ thông số train chuẩn (KHÔNG bật augmentation thủ công).

In [4]:
import os, shutil
os.environ['WANDB_MODE'] = 'disabled'

RUN_NAME = 'vehicle_yolov8s_v6'

results = model.train(
    data=YAML_PATH,
    imgsz=640,
    device=[0, 1],          # 2 GPU (T4x2). Neu 1 GPU -> doi thanh device=0
    batch=BATCH,
    workers=8,
    amp=True,
    epochs=150,
    pretrained=True,
    optimizer='auto',
    lr0=0.01,
    cos_lr=True,
    close_mosaic=10,
    seed=42,
    deterministic=True,
    save_period=10,
    name=RUN_NAME,
)

# Danh gia tren tap val
metrics = model.val()
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'mAP50   : {metrics.box.map50:.4f}')

# Nen ket qua de tai ve
shutil.make_archive(f'/kaggle/working/{RUN_NAME}_results', 'zip',
                    f'/kaggle/working/runs/detect/{RUN_NAME}')
print('[OK] Done ->', RUN_NAME)

Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicle_yolov8s_v6, 

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:99: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1487.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 6.0it/s 3.5s
                   all        324        466      0.634      0.462      0.521      0.284

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/150      1.37G       1.23      1.498       1.54          9        640: 100% ━━━━━━━━━━━━ 420/420 7.8it/s 53.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 11.2it/s 1.9s
                   all        324        466       0.65      0.356      0.445      0.248

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/150      1.37G      1.234      1.485      1.559          9        640: 100% ━━━━━━━━━━━━ 420/420 7.9it/s 53.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 11.4it/s 1.8s
                   all        324        46